# Flow

In [ ]:
✅ Tokenization
✅ Embeddings
✅ Positional Encoding
✅ Self-Attention
✅ Multi-Head Attention
✅ FFN
✅ LayerNorm
✅ Encoder Block
✅ Encoder
✅ Causal / Masked Self-Attention
✅ Decoder MHA

        ↓ NEXT

⬜ Cross-Attention ---> This NoteBook
⬜ Decoder Block
⬜ Full Transformer
⬜ Teacher Forcing
⬜ Output Projection + Softmax
⬜ Cross-Entropy Loss
⬜ Backpropagation
⬜ Tiny Transformer Training
⬜ Text Generation
⬜ GPT / Decoder-only Architecture
⬜ BPE Tokenization
⬜ RoPE
⬜ RMSNorm
⬜ SwiGLU
⬜ GQA / MQA
⬜ KV Cache
⬜ Tiny GPT
⬜ Fine-tuning
⬜ LoRA
⬜ RAG
⬜ Production LLM System

# Problem stmt

# Cross attention Flow

# Function

In [5]:
import numpy as np
import pandas as pd
import math

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def cross_attention(
    decoder_input,
    encoder_output,
    Wq,
    Wk,
    Wv,
    Wo,
    num_heads
):
    Q=decoder_input @ Wq
    K=encoder_output @ Wk
    V=encoder_output @ Wv

    batch_size, seq_length, d_model = Q.shape
    batch_size_en, seq_length_en, d_model_en = K.shape
    # define d_head
    d_head=d_model//num_heads

    Q_head=Q.reshape(batch_size,seq_length,num_heads,d_head)
    K_head=K.reshape(batch_size_en,seq_length_en,num_heads,d_head)
    V_head=V.reshape(batch_size_en,seq_length_en,num_heads,d_head)

    Q_head=Q_head.transpose(0,2,1,3)
    K_head=K_head.transpose(0,2,1,3)
    V_head=V_head.transpose(0,2,1,3)

    # q and kt dot product
    score=Q_head @ K_head.transpose(0,1,3,2)
    score=score/np.sqrt(d_head)

    # apply softmax
    score_soft=softmax(score,axis=-1)
    attentions=score_soft @ V_head

    head_output=attentions.transpose(0,2,1,3)
    head_output=head_output.reshape(batch_size, seq_length, d_model)
    
    output=head_output@Wo
    return output,score_soft
    

# Run code

In [3]:
decoder_input = np.array([[
    [1.0, 0.0],
    [0.0, 1.0]
]])

encoder_output = np.array([[
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
]])

Wq = np.eye(2)
Wk = np.eye(2)
Wv = np.eye(2)
Wo = np.eye(2)

num_heads = 1

In [6]:
cross_attention(decoder_input,encoder_output,Wq,Wk,Wv,Wo,num_heads)

(array([[[0.80222419, 0.59888791],
         [0.59888791, 0.80222419]]]),
 array([[[[0.40111209, 0.19777581, 0.40111209],
          [0.19777581, 0.40111209, 0.40111209]]]]))